Step 1: Project Initialization and Dependency Setup

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 2.8 MB/s eta 0:00:00


In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step 2: Data Loading and Preprocessing

In [ ]:
import pandas as pd

rag_df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/results/rag_documents.csv")
documents = rag_df["RAG_Document"].tolist()

print("Total RAG documents:", len(documents))
print(documents[0])

Total RAG documents: 3378
Block ID        : blk_8462687553742484299
Label           : Fail
Anomaly Type(s) : duplicate_pattern

--- Details ---
Event Sequence  : E5 -> E5 -> E5 -> E22 -> E11 -> E9 -> E11 -> E9 -> E11 -> E9 -> E26 -> E26 -> E26 -> E23 -> E23 -> E23 -> E21 -> E20 -> E21 -> E21
Readable Form   : [*]Receiving block[*]src:[*]dest:[*] -> [*]Receiving block[*]src:[*]dest:[*] -> [*]Receiving block[*]src:[*]dest:[*] -> [*]BLOCK* NameSystem[*]allocateBlock:[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]PacketResponder[*]for block[*]terminating[*] -> [*]Received block[*]of size[*]from[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] -> [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] 

Step 3: Semantic Embedding Generation

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (3378, 384)


Step 4: Vector Database Indexing with FAISS

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors stored:", index.ntotal)

Vectors stored: 3378


Step 5: Retrieval and LLM Analysis Logic

In [ ]:
def retrieve_similar_anomalies(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "document": documents[idx],
            "distance": float(dist)   # lower = more similar
        })
    return results

In [ ]:
from groq import Groq

client = Groq(api_key="your_api_key")

In [ ]:
def generate_root_cause(context):
    SYSTEM_PROMPT = """You are an expert in Hadoop Distributed File System (HDFS) operations and log analysis.

You will be given retrieved anomaly cases. Each case has five sections:
- Details: raw event data for the block
- Summary: what type of anomaly occurred
- Root Cause: why it likely occurred
- Important: severity and impact notes

Use these sections as evidence. Be specific — reference the actual event sequences and counts."""

    USER_PROMPT = f"""The following anomaly cases were retrieved as similar to the query:

{context}

Based on the evidence above, provide:
1. Most likely root cause (cite specific events or patterns from the cases)
2. Anomaly category: system / network / configuration
3. Confidence level: high / medium / low — and why
4. Mitigation steps (ordered by priority)"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT}
            ]
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"[ERROR] Groq API call failed: {e}"